# MotionEmo — A/T/S 3D Interactive Upload Demo

Notebook này dùng để upload **1 audio ngắn**, chạy pipeline xử lý nhẹ và **Pha 2** để lấy chuỗi `Activation / Tension / Stability`:

1. **3D Context A/T/(1-S)**: toàn bộ audio trong không gian `Activation – Tension – Instability`, vùng minh họa được tô nổi bật.
2. **3D A/T/S State Trajectory**: quỹ đạo trạng thái trực tiếp theo ba trục `Activation – Tension – Stability`.
3. **3D A/T/(1-S) Salience Cube**: không gian nổi bật động học, trong đó `Instability = 1 - Stability` để cả ba trục cùng chiều với dynamic salience.


In [ ]:
# ============================================================
# CELL 1 — Mount Drive và cài thư viện
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch torchaudio numpy pandas scipy soundfile librosa pyloudnorm matplotlib tqdm scikit-learn umap-learn noisereduce plotly

Mounted at /content/drive


In [ ]:
# ============================================================
# CELL 2 — Import thư viện
# ============================================================
import os
import json
import math
import random
import warnings
import hashlib
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple
from contextlib import nullcontext

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import soundfile as sf
import librosa
import pyloudnorm as pyln

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib as mpl
# Plotly dùng cho hình 3D tương tác.
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.decomposition import PCA

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False

try:
    import noisereduce as nr
    NOISEREDUCE_AVAILABLE = True
except Exception:
    NOISEREDUCE_AVAILABLE = False

from IPython.display import display, Audio, HTML, Markdown
from google.colab import files

warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CELL 3 — Cấu hình đường dẫn và tham số demo
# ============================================================
DRIVE_ROOT = Path('/content/drive/MyDrive')

PHASE1_CKPT = DRIVE_ROOT / 'motionemo_v10_phase1/checkpoints/best_phase1_v10.pt'
PHASE2_CKPT = DRIVE_ROOT / 'motionemo_v10_phase2/checkpoints/best_phase2_state.pt'

LOCAL_DATA_ROOT = Path('/content/processed_v10_safe')
DRIVE_DATA_ROOT = DRIVE_ROOT / 'data/processed_v10_safe'
DATA_ROOT = LOCAL_DATA_ROOT if (LOCAL_DATA_ROOT / 'all_metadata.csv').exists() else DRIVE_DATA_ROOT
METADATA_PATH = DATA_ROOT / 'all_metadata.csv'
PRECOMPUTED_DIR = DATA_ROOT / 'precomputed'

OUTPUT_ROOT = DRIVE_ROOT / 'motionemo_v10_ats_3d_upload_demo_optimized'
CACHE_ROOT = OUTPUT_ROOT / '_cache'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP_ENABLED = DEVICE == 'cuda'

STATE_NAMES = ['activation', 'tension', 'stability']
STATE_DISPLAY_NAMES = {
    'activation': 'Vocal Activation',
    'tension': 'Vocal Tension',
    'stability': 'Vocal Stability',
}
STATE_LABEL_NAMES = ['low', 'mid', 'high']
STATE_SCORE_LEVELS = np.array([0.0, 0.5, 1.0], dtype=np.float32)

COLORS = {
    'activation': '#2F80ED',
    'tension': '#E76F51',
    'stability': '#2A9D8F',
    'transition': '#7B2CBF',
    'stable': '#2A9D8F',
    'unstable': '#E76F51',
    'mixed': '#6C757D',
    'grid': '#DEE2E6',
    'text': '#212529',
}

AUDIO_CFG = {
    'sample_rate': 16000,
    'n_mels': 80,
    'n_fft': 512,
    'win_length': 400,
    'hop_length': 160,
    'f_min': 50.0,
    'f_max': 8000.0,
    'f0_min': 50.0,
    'f0_max': 500.0,
    'top_db': 80.0,
    'target_lufs': -23.0,
    'peak_guard': 0.98,
}

WINDOW_CFG = {
    'window_sec': 2.5,
    'hop_sec': 0.5,
    'min_window_sec': 1.2,
    'max_frames': 500,
}

DEMO_CFG = {
    # Reference analysis.
    'reference_split': 'test',
    'max_reference_samples': 1000,
    'reference_random_seed': 42,
    'use_umap': True,

    # Audio demo.
    'smooth_window': 5,
    'use_light_denoise': False,
    'use_silence_trim': False,  # Giữ timeline gốc cho minh họa; đổi True nếu chỉ muốn bỏ lặng đầu/cuối.

    # Evidence rules.
    'high_tension_threshold': 0.65,
    'low_stability_threshold': 0.35,
    'stable_tension_threshold': 0.25,
    'stable_stability_threshold': 0.70,
    'rapid_transition_quantile': 0.85,

    # Long audio visualization.
    'max_windows_for_dense_trajectory': 70,
    'segment_sec_for_long_audio': 5.0,
}

# Chế độ audio demo.
USE_PREDEFINED_AUDIO = False
PREDEFINED_AUDIO_PATHS = [
    # DRIVE_ROOT / 'demo_audio/angry.mp3',
    # DRIVE_ROOT / 'demo_audio/calm_but_abit_hesitation.mp3',
]

print('Device:', DEVICE)
print('AMP:', AMP_ENABLED)
print('Data root:', DATA_ROOT)
print('Metadata exists:', METADATA_PATH.exists())
print('Precomputed exists:', PRECOMPUTED_DIR.exists())
print('Phase 2 checkpoint exists:', PHASE2_CKPT.exists())
print('UMAP available:', UMAP_AVAILABLE)
print('Output root:', OUTPUT_ROOT)

Device: cuda
AMP: True
Data root: /content/drive/MyDrive/data/processed_v10_safe
Metadata exists: True
Precomputed exists: True
Phase 2 checkpoint exists: True
UMAP available: True
Output root: /content/drive/MyDrive/motionemo_v10_ats_3d_upload_demo_optimized


In [ ]:
# ============================================================
# CELL 4 — Helper chung
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


def autocast_context():
    if AMP_ENABLED and DEVICE == 'cuda':
        return torch.amp.autocast('cuda', enabled=True)
    return nullcontext()


def safe_filename_stem(path: Path) -> str:
    return Path(path).stem.replace(' ', '_').replace('/', '_').replace('\\', '_')


def make_padding_mask(lengths: torch.Tensor, max_len: int) -> torch.Tensor:
    idx = torch.arange(max_len, device=lengths.device).unsqueeze(0)
    return idx >= lengths.unsqueeze(1)


def rolling_smooth(series: pd.Series, window: int = 5) -> pd.Series:
    return series.rolling(window=window, center=True, min_periods=1).mean()


def safe_zscore_np(x: np.ndarray, mask: Optional[np.ndarray] = None, eps: float = 1e-8) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    if x.size == 0:
        return x.astype(np.float32)
    if mask is not None:
        mask = np.asarray(mask).astype(bool)
        if mask.sum() > 1:
            mu = float(x[mask].mean())
            sd = float(x[mask].std())
        else:
            mu = float(x.mean())
            sd = float(x.std())
    else:
        mu = float(x.mean())
        sd = float(x.std())
    if not np.isfinite(sd) or sd < eps:
        sd = 1.0
    return ((x - mu) / (sd + eps)).astype(np.float32)


def align_1d_to_T(x: np.ndarray, T: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if len(x) == T:
        return x
    if len(x) > T:
        return x[:T]
    if len(x) == 0:
        return np.zeros(T, dtype=np.float32)
    return np.concatenate([x, np.full(T - len(x), x[-1], dtype=np.float32)], axis=0)


def savefig(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=220, bbox_inches='tight', facecolor='white')
    print('Saved:', path)


def checkpoint_fingerprint(path: Path) -> str:
    if not path.exists():
        return 'missing'
    text = f'{path.name}_{path.stat().st_size}_{int(path.stat().st_mtime)}'
    return hashlib.md5(text.encode('utf-8')).hexdigest()[:10]


def display_note(title: str, body: str):
    display(HTML(f"""
    <div style='font-family:Arial; padding:14px; border-radius:12px; border:1px solid #dee2e6; background:#ffffff; color:#212529;'>
      <h3 style='margin-top:0'>{title}</h3>
      <div style='font-size:14px; line-height:1.55'>{body}</div>
    </div>
    """))

In [ ]:
# ============================================================
# CELL 5 — Định nghĩa backbone và state heads
# ============================================================
class MotionEmoBackboneV10(nn.Module):
    def __init__(self, n_mels=80, embed_dim=512, num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.prosody_encoder = nn.Sequential(
            nn.Conv1d(3, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=5, padding=2), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.mel_encoder = nn.Sequential(
            nn.Conv1d(n_mels, 256, kernel_size=5, padding=2), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.fusion_proj = nn.Sequential(nn.Linear(512, embed_dim), nn.LayerNorm(embed_dim))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.attn_pool = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 4), nn.Tanh(), nn.Linear(embed_dim // 4, 1)
        )

    def forward(self, mel, f0, energy, srate, lengths: Optional[torch.Tensor] = None):
        B, _, T = mel.shape
        prosody_in = torch.stack([f0, energy, srate], dim=1)
        p_feat = self.prosody_encoder(prosody_in)
        m_feat = self.mel_encoder(mel)
        fused = torch.cat([p_feat, m_feat], dim=1)
        x = self.fusion_proj(fused.transpose(1, 2))
        pad_mask = make_padding_mask(lengths, T) if lengths is not None else None
        H_frame = self.transformer(x, src_key_padding_mask=pad_mask)
        attn_logits = self.attn_pool(H_frame).squeeze(-1)
        if pad_mask is not None:
            attn_logits = attn_logits.masked_fill(pad_mask, -1e4)
        attn = torch.softmax(attn_logits, dim=1).unsqueeze(-1)
        H_utt = (attn * H_frame).sum(dim=1)
        return H_frame, H_utt


class StateHead(nn.Module):
    def __init__(self, embed_dim=512, hidden_dim=256, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, H_utt):
        return self.net(H_utt)

In [ ]:
# ============================================================
# CELL 6 — Tải checkpoint Pha 2
# ============================================================
def load_checkpoint(path: Path):
    return torch.load(path, map_location=DEVICE, weights_only=False)


def infer_model_config(ckpt: Dict[str, Any]):
    cfg = ckpt.get('cfg', {})
    model_cfg = cfg.get('model', {}) if isinstance(cfg, dict) else {}
    return {
        'embed_dim': int(model_cfg.get('embedding_dim', 512)),
        'num_heads': int(model_cfg.get('num_heads', 8)),
        'num_layers': int(model_cfg.get('num_layers', 6)),
        'dropout': float(model_cfg.get('dropout', 0.1)),
        'state_hidden': int(model_cfg.get('state_hidden', 256)),
        'num_state_classes': int(model_cfg.get('num_state_classes', 3)),
    }


def build_backbone_from_ckpt(ckpt: Dict[str, Any]):
    mc = infer_model_config(ckpt)
    model = MotionEmoBackboneV10(
        n_mels=AUDIO_CFG['n_mels'],
        embed_dim=mc['embed_dim'],
        num_heads=mc['num_heads'],
        num_layers=mc['num_layers'],
        dropout=mc['dropout'],
    ).to(DEVICE)
    model.load_state_dict(ckpt['backbone'], strict=True)
    model.eval()
    return model, mc


if not PHASE2_CKPT.exists():
    raise FileNotFoundError(f'Missing Phase 2 checkpoint: {PHASE2_CKPT}')

phase2_ckpt = load_checkpoint(PHASE2_CKPT)
phase2_backbone, model_cfg = build_backbone_from_ckpt(phase2_ckpt)

activation_head = StateHead(model_cfg['embed_dim'], model_cfg['state_hidden'], model_cfg['num_state_classes']).to(DEVICE)
tension_head = StateHead(model_cfg['embed_dim'], model_cfg['state_hidden'], model_cfg['num_state_classes']).to(DEVICE)
stability_head = StateHead(model_cfg['embed_dim'], model_cfg['state_hidden'], model_cfg['num_state_classes']).to(DEVICE)
activation_head.load_state_dict(phase2_ckpt['activation_head'])
tension_head.load_state_dict(phase2_ckpt['tension_head'])
stability_head.load_state_dict(phase2_ckpt['stability_head'])
activation_head.eval(); tension_head.eval(); stability_head.eval()

# Notebook này chỉ dùng Pha 2 để lấy chuỗi A/T/S.
# Pha 1 checkpoint không cần tải trong notebook này.
print('Loaded Phase 2 checkpoint:', PHASE2_CKPT)
print('Phase 2 epoch:', phase2_ckpt.get('epoch'))
print('Phase 2 best val loss:', phase2_ckpt.get('best_val_loss'))
print('Phase 2 best state acc:', phase2_ckpt.get('best_state_acc'))
print('Model config:', model_cfg)

Loaded Phase 2 checkpoint: /content/drive/MyDrive/motionemo_v10_phase2/checkpoints/best_phase2_state.pt
Phase 2 epoch: 25
Phase 2 best val loss: 0.8443085433842542
Phase 2 best state acc: 0.868068068068068
Model config: {'embed_dim': 512, 'num_heads': 8, 'num_layers': 6, 'dropout': 0.1, 'state_hidden': 256, 'num_state_classes': 3}


In [ ]:
# ============================================================
# CELL 7 — Pipeline audio + trích đặc trưng + inference Pha 2
# ============================================================
def load_audio_mono(path: Path, sr: int = 16000) -> Tuple[np.ndarray, int, int]:
    wav, native_sr = sf.read(str(path), always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float32)
    wav = np.nan_to_num(wav, nan=0.0, posinf=0.0, neginf=0.0)
    if native_sr != sr:
        wav = librosa.resample(wav, orig_sr=native_sr, target_sr=sr).astype(np.float32)
    return wav, sr, native_sr


def estimate_lufs(wav: np.ndarray, sr: int) -> float:
    if len(wav) < int(0.4 * sr):
        return float('-inf')
    peak = float(np.max(np.abs(wav))) if len(wav) else 0.0
    rms = float(np.sqrt(np.mean(wav ** 2) + 1e-12)) if len(wav) else 0.0
    if peak < 1e-5 or rms < 1e-6:
        return float('-inf')
    try:
        meter = pyln.Meter(sr)
        val = meter.integrated_loudness(wav.astype(np.float64))
        return float(val) if np.isfinite(val) else float('-inf')
    except Exception:
        return float('-inf')


def normalize_lufs_light(wav: np.ndarray, sr: int, target_lufs=-23.0):
    lufs = estimate_lufs(wav, sr)
    if not np.isfinite(lufs) or lufs < -60:
        return wav.astype(np.float32)
    gain_db = np.clip(target_lufs - lufs, -24.0, 24.0)
    y = wav * (10 ** (gain_db / 20.0))
    peak = np.max(np.abs(y)) if len(y) else 0.0
    if peak > AUDIO_CFG['peak_guard']:
        y = y * (AUDIO_CFG['peak_guard'] / (peak + 1e-8))
    return y.astype(np.float32)


def preprocess_demo_audio(wav: np.ndarray, sr: int):
    y = normalize_lufs_light(wav, sr, AUDIO_CFG['target_lufs'])
    if DEMO_CFG['use_silence_trim'] and len(y) > sr:
        intervals = librosa.effects.split(y, top_db=35, frame_length=AUDIO_CFG['win_length'], hop_length=AUDIO_CFG['hop_length'])
        if len(intervals):
            start = max(0, int(intervals[0][0]) - int(0.15 * sr))
            end = min(len(y), int(intervals[-1][1]) + int(0.15 * sr))
            if (end - start) > int(WINDOW_CFG['min_window_sec'] * sr):
                y = y[start:end]
    return y.astype(np.float32)


def extract_v10_features_from_waveform(wav: np.ndarray, sr: int) -> Dict[str, torch.Tensor]:
    wav = np.asarray(wav, dtype=np.float32)
    mel_power = librosa.feature.melspectrogram(
        y=wav,
        sr=sr,
        n_fft=AUDIO_CFG['n_fft'],
        hop_length=AUDIO_CFG['hop_length'],
        win_length=AUDIO_CFG['win_length'],
        n_mels=AUDIO_CFG['n_mels'],
        fmin=AUDIO_CFG['f_min'],
        fmax=AUDIO_CFG['f_max'],
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel_power, ref=np.max, top_db=AUDIO_CFG['top_db']).astype(np.float32)
    mel = safe_zscore_np(mel_db)
    T = mel.shape[1]

    rms = librosa.feature.rms(y=wav, frame_length=AUDIO_CFG['win_length'], hop_length=AUDIO_CFG['hop_length'], center=True)[0].astype(np.float32)
    rms = align_1d_to_T(rms, T)
    zcr = librosa.feature.zero_crossing_rate(y=wav, frame_length=AUDIO_CFG['win_length'], hop_length=AUDIO_CFG['hop_length'], center=True)[0].astype(np.float32)
    zcr = align_1d_to_T(zcr, T)

    try:
        f0 = librosa.yin(
            wav,
            fmin=AUDIO_CFG['f0_min'],
            fmax=AUDIO_CFG['f0_max'],
            sr=sr,
            frame_length=AUDIO_CFG['win_length'],
            hop_length=AUDIO_CFG['hop_length'],
        ).astype(np.float32)
    except Exception:
        f0 = np.zeros(T, dtype=np.float32)
    f0 = align_1d_to_T(np.nan_to_num(f0, nan=0.0), T)

    rms_p95 = float(np.percentile(rms, 95)) + 1e-8
    voiced_mask = (rms / rms_p95 > 0.03) & (f0 > 0)
    f0_raw = f0.copy()
    f0_raw[~voiced_mask] = 0.0
    f0_norm = np.zeros_like(f0_raw, dtype=np.float32)
    if voiced_mask.sum() > 1:
        f0_norm[voiced_mask] = safe_zscore_np(f0_raw[voiced_mask])

    return {
        'mel': torch.from_numpy(mel.astype(np.float32)),
        'f0': torch.from_numpy(f0_norm.astype(np.float32)),
        'energy': torch.from_numpy(safe_zscore_np(rms).astype(np.float32)),
        'srate': torch.from_numpy(safe_zscore_np(zcr).astype(np.float32)),
        'num_frames': T,
        'energy_raw': torch.from_numpy(rms.astype(np.float32)),
        'f0_raw': torch.from_numpy(f0_raw.astype(np.float32)),
        'zcr_raw': torch.from_numpy(zcr.astype(np.float32)),
    }


def make_sliding_windows(wav: np.ndarray, sr: int) -> List[Tuple[int, int, np.ndarray]]:
    win = int(WINDOW_CFG['window_sec'] * sr)
    hop = int(WINDOW_CFG['hop_sec'] * sr)
    min_len = int(WINDOW_CFG['min_window_sec'] * sr)
    windows = []
    if len(wav) <= win:
        if len(wav) >= min_len:
            windows.append((0, len(wav), wav.copy()))
        return windows
    start = 0
    while start < len(wav):
        end = start + win
        chunk = wav[start:min(end, len(wav))]
        if len(chunk) >= min_len:
            windows.append((start, min(end, len(wav)), chunk.copy()))
        if end >= len(wav):
            break
        start += hop
    return windows


@torch.no_grad()
def infer_demo_windows(features: List[Dict[str, torch.Tensor]], batch_size=32):
    rows, embs = [], []
    for start in tqdm(range(0, len(features), batch_size), desc='Infer demo windows'):
        chunk = features[start:start + batch_size]
        max_len = max(x['num_frames'] for x in chunk)
        mels, f0s, energies, srates, lengths = [], [], [], [], []
        for item in chunk:
            L = item['num_frames']
            pad = max_len - L
            mels.append(F.pad(item['mel'], (0, pad)).unsqueeze(0))
            f0s.append(F.pad(item['f0'], (0, pad)).unsqueeze(0))
            energies.append(F.pad(item['energy'], (0, pad)).unsqueeze(0))
            srates.append(F.pad(item['srate'], (0, pad)).unsqueeze(0))
            lengths.append(L)
        mel = torch.cat(mels).to(DEVICE)
        f0 = torch.cat(f0s).to(DEVICE)
        energy = torch.cat(energies).to(DEVICE)
        srate = torch.cat(srates).to(DEVICE)
        lengths_t = torch.tensor(lengths, dtype=torch.long, device=DEVICE)
        with autocast_context():
            _, h = phase2_backbone(mel, f0, energy, srate, lengths=lengths_t)
            ap = F.softmax(activation_head(h), dim=-1)
            tp = F.softmax(tension_head(h), dim=-1)
            sp = F.softmax(stability_head(h), dim=-1)
        embs.append(h.float().cpu().numpy())
        for i in range(len(chunk)):
            a = ap[i].float().cpu().numpy()
            t = tp[i].float().cpu().numpy()
            s = sp[i].float().cpu().numpy()
            rows.append({
                'activation_score': float((a * STATE_SCORE_LEVELS).sum()),
                'tension_score': float((t * STATE_SCORE_LEVELS).sum()),
                'stability_score': float((s * STATE_SCORE_LEVELS).sum()),
                'activation_pred': int(a.argmax()),
                'tension_pred': int(t.argmax()),
                'stability_pred': int(s.argmax()),
            })
    return pd.DataFrame(rows), np.concatenate(embs, axis=0)


def add_timeline_metrics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for state in STATE_NAMES:
        df[f'{state}_score_smooth'] = rolling_smooth(df[f'{state}_score'], DEMO_CFG['smooth_window'])
        df[f'd_{state}'] = df[f'{state}_score_smooth'].diff().fillna(0.0)
    df['state_velocity'] = np.sqrt(df['d_activation']**2 + df['d_tension']**2 + df['d_stability']**2)
    return df


def summarize_before_after(df: pd.DataFrame) -> Dict[str, float]:
    n = len(df)
    first = df.iloc[:max(1, n // 3)]
    last = df.iloc[max(0, 2 * n // 3):]
    return {
        'first_third_tension': float(first['tension_score_smooth'].mean()),
        'last_third_tension': float(last['tension_score_smooth'].mean()),
        'delta_tension_last_minus_first': float(last['tension_score_smooth'].mean() - first['tension_score_smooth'].mean()),
        'first_third_stability': float(first['stability_score_smooth'].mean()),
        'last_third_stability': float(last['stability_score_smooth'].mean()),
        'delta_stability_last_minus_first': float(last['stability_score_smooth'].mean() - first['stability_score_smooth'].mean()),
    }


def analyze_audio_file(audio_path: Path):
    wav, sr, native_sr = load_audio_mono(audio_path, AUDIO_CFG['sample_rate'])
    wav_p = preprocess_demo_audio(wav, sr)
    windows = make_sliding_windows(wav_p, sr)
    if not windows:
        raise RuntimeError('Audio too short after preprocessing.')
    feats, times = [], []
    for i, (st, en, chunk) in enumerate(tqdm(windows, desc=f'Extract features: {audio_path.name}')):
        feats.append(extract_v10_features_from_waveform(chunk, sr))
        times.append({'window_id': i, 'start_sec': st / sr, 'end_sec': en / sr, 'center_sec': (st + en) / (2 * sr)})
    pred, emb = infer_demo_windows(feats)
    df = pd.concat([pd.DataFrame(times), pred], axis=1)
    df = add_timeline_metrics(df)
    return wav, wav_p, sr, df, emb

In [ ]:
# ============================================================
# CELL 8 — Upload 1 audio ngắn
# ============================================================
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Bạn chưa upload audio.')

audio_path = Path(list(uploaded.keys())[0])
print('Uploaded:', audio_path)

Saving calm_but_abit_hesitation.mp3 to calm_but_abit_hesitation.mp3
Uploaded: calm_but_abit_hesitation.mp3


In [ ]:
# ============================================================
# CELL 9 — Chạy pipeline + Pha 2 để lấy chuỗi A/T/S
# ============================================================
prefix = safe_filename_stem(audio_path)
out_dir = OUTPUT_ROOT / (prefix + '_ats3d_optimized')
out_dir.mkdir(parents=True, exist_ok=True)

wav_original, wav_processed, sr, ats_df, phase2_emb = analyze_audio_file(audio_path)

# Bổ sung các cột phục vụ hình 3D và diễn giải.
ats_df['instability_score'] = 1.0 - ats_df['stability_score']
ats_df['instability_score_smooth'] = 1.0 - ats_df['stability_score_smooth']
ats_df['dynamic_salience'] = (
    ats_df['activation_score_smooth'] +
    ats_df['tension_score_smooth'] +
    ats_df['instability_score_smooth']
) / 3.0

sf.write(str(out_dir / f'{prefix}_processed.wav'), wav_processed, sr)
ats_df.to_csv(out_dir / f'{prefix}_ats_timeline.csv', index=False)
np.save(out_dir / f'{prefix}_phase2_embeddings.npy', phase2_emb)

print('Original duration:', round(len(wav_original) / sr, 3), 's')
print('Processed duration:', round(len(wav_processed) / sr, 3), 's')
print('Number of windows:', len(ats_df))
print('Saved timeline:', out_dir / f'{prefix}_ats_timeline.csv')

display(Markdown(f'## Audio đã upload: `{audio_path.name}`'))
display(Audio(wav_processed, rate=sr))

Extract features: calm_but_abit_hesitation.mp3:   0%|          | 0/48 [00:00<?, ?it/s]

Infer demo windows:   0%|          | 0/2 [00:00<?, ?it/s]

Original duration: 25.99 s
Processed duration: 25.99 s
Number of windows: 48
Saved timeline: /content/drive/MyDrive/motionemo_v10_ats_3d_upload_demo_optimized/calm_but_abit_hesitation_ats3d_optimized/calm_but_abit_hesitation_ats_timeline.csv


## Audio đã upload: `calm_but_abit_hesitation.mp3`

In [ ]:
# ============================================================
# CELL 10 — Tóm tắt nhanh A/T/S
# ============================================================
def summarize_ats(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    specs = [
        ('activation', 'activation_score_smooth'),
        ('tension', 'tension_score_smooth'),
        ('stability', 'stability_score_smooth'),
        ('instability', 'instability_score_smooth'),
        ('dynamic_salience', 'dynamic_salience'),
        ('state_velocity', 'state_velocity'),
    ]
    for name, col in specs:
        first = df.iloc[:max(1, len(df)//3)][col].mean()
        last = df.iloc[max(0, 2*len(df)//3):][col].mean()
        rows.append({
            'measure': name,
            'mean': float(df[col].mean()),
            'std': float(df[col].std()),
            'min': float(df[col].min()),
            'max': float(df[col].max()),
            'first_third': float(first),
            'last_third': float(last),
            'delta_last_minus_first': float(last - first),
        })
    return pd.DataFrame(rows)

ats_summary = summarize_ats(ats_df)
ats_summary.to_csv(out_dir / f'{prefix}_ats_summary.csv', index=False)
display(ats_summary)

peak_t = ats_df.loc[ats_df['tension_score_smooth'].idxmax()]
min_s = ats_df.loc[ats_df['stability_score_smooth'].idxmin()]
max_v = ats_df.loc[ats_df['state_velocity'].idxmax()]
print(f"Peak tension: {peak_t['center_sec']:.2f}s | value={peak_t['tension_score_smooth']:.3f}")
print(f"Minimum stability: {min_s['center_sec']:.2f}s | value={min_s['stability_score_smooth']:.3f}")
print(f"Largest transition: {max_v['center_sec']:.2f}s | velocity={max_v['state_velocity']:.3f}")

,measure,mean,std,min,max,first_third,last_third,delta_last_minus_first
0,activation,0.851069,0.148319,0.507052,0.998055,0.782286,0.923842,0.141556
1,tension,0.313810,0.301022,0.000681,0.994778,0.601491,0.134054,-0.467436
2,stability,0.600840,0.335313,0.017803,0.998136,0.253203,0.799819,0.546616
3,instability,0.399160,0.335313,0.001864,0.982197,0.746797,0.200181,-0.546616
4,dynamic_salience,0.521346,0.211616,0.219766,0.939187,0.710191,0.419359,-0.290832
5,state_velocity,0.110385,0.077522,0.000000,0.291378,0.136788,0.066802,-0.069985


Peak tension: 8.25s | value=0.995
Minimum stability: 8.25s | value=0.018
Largest transition: 3.25s | velocity=0.291


In [ ]:
# ============================================================
# CELL 11 — Tự động gợi ý khoảng thời gian tốt nhất để minh họa
# ============================================================
def recommend_interpretive_interval(
    df: pd.DataFrame,
    total_duration: float,
    min_sec: float = 4.0,
    max_sec: float = 7.0,
    step_sec: float = 0.25,
) -> Dict[str, Any]:
    """Chọn đoạn tốt nhất để minh họa A/T/S.

    Tiêu chí ưu tiên:
    - dynamic salience trung bình cao;
    - có chuyển trạng thái nhanh;
    - có biên độ biến thiên trong A/T/S;
    - đủ dài để nghe và giải thích trong luận văn/slide.
    """
    if len(df) == 0:
        return {'start': 0.0, 'end': min(total_duration, max_sec), 'score': 0.0, 'reason': 'empty dataframe'}

    if total_duration <= max_sec + 1.0:
        return {
            'start': 0.0,
            'end': float(total_duration),
            'score': float(df['dynamic_salience'].mean()),
            'reason': 'Audio ngắn; dùng toàn bộ đoạn để giữ mạch thời gian.'
        }

    d = df.copy()
    vel = d['state_velocity'].to_numpy(dtype=float)
    if np.nanmax(vel) - np.nanmin(vel) > 1e-8:
        d['state_velocity_norm'] = (vel - np.nanmin(vel)) / (np.nanmax(vel) - np.nanmin(vel) + 1e-8)
    else:
        d['state_velocity_norm'] = 0.0

    durations = sorted(set([min(max_sec, total_duration), 6.0, 5.0, 4.0]), reverse=True)
    best = None
    for dur in durations:
        if dur > total_duration:
            continue
        starts = np.arange(0.0, max(0.0, total_duration - dur) + 1e-6, step_sec)
        for st in starts:
            en = st + dur
            sub = d[(d['center_sec'] >= st) & (d['center_sec'] <= en)]
            if len(sub) < 4:
                continue
            sal_mean = float(sub['dynamic_salience'].mean())
            vel_max = float(sub['state_velocity_norm'].max())
            amp_sal = float(sub['dynamic_salience'].max() - sub['dynamic_salience'].min())
            amp_ats = float(np.mean([
                sub['activation_score_smooth'].max() - sub['activation_score_smooth'].min(),
                sub['tension_score_smooth'].max() - sub['tension_score_smooth'].min(),
                sub['stability_score_smooth'].max() - sub['stability_score_smooth'].min(),
            ]))
            tension_mean = float(sub['tension_score_smooth'].mean())
            instability_mean = float(sub['instability_score_smooth'].mean())
            score = (
                0.38 * sal_mean +
                0.25 * vel_max +
                0.20 * amp_ats +
                0.10 * amp_sal +
                0.04 * tension_mean +
                0.03 * instability_mean
            )
            item = {
                'start': float(st), 'end': float(en), 'duration': float(dur), 'score': float(score),
                'salience_mean': sal_mean, 'velocity_peak_norm': vel_max,
                'ats_amplitude': amp_ats, 'salience_amplitude': amp_sal,
                'tension_mean': tension_mean, 'instability_mean': instability_mean,
            }
            if best is None or item['score'] > best['score']:
                best = item

    if best is None:
        return {
            'start': 0.0,
            'end': min(float(total_duration), max_sec),
            'score': float(d['dynamic_salience'].mean()),
            'reason': 'Không đủ cửa sổ ứng viên; dùng đoạn đầu.'
        }

    best['reason'] = (
        'Đoạn được chọn vì có tổng hợp tốt giữa dynamic salience, chuyển trạng thái nhanh '
        'và biên độ biến thiên A/T/S. Đây là vùng phù hợp để minh họa bằng timeline và 3D trajectory.'
    )
    return best

best_interval = recommend_interpretive_interval(ats_df, total_duration=len(wav_processed)/sr)
best_start, best_end = best_interval['start'], best_interval['end']
selected_df = ats_df[(ats_df['center_sec'] >= best_start) & (ats_df['center_sec'] <= best_end)].copy()
if len(selected_df) < 4:
    selected_df = ats_df.copy()
    best_start, best_end = float(ats_df['start_sec'].min()), float(ats_df['end_sec'].max())

interval_info = pd.DataFrame([best_interval])
interval_info.to_csv(out_dir / f'{prefix}_recommended_interval.csv', index=False)
display(interval_info)

s_i = max(0, int(best_start * sr))
e_i = min(len(wav_processed), int(best_end * sr))
interval_audio = wav_processed[s_i:e_i]
sf.write(str(out_dir / f'{prefix}_recommended_interval_{best_start:.1f}_{best_end:.1f}s.wav'), interval_audio, sr)

display(Markdown(
    f"### Khoảng minh họa đề xuất: **{best_start:.2f}s – {best_end:.2f}s**\n"
    f"{best_interval.get('reason', '')}\n\n"
    "Các hình 3D phía dưới sẽ dùng khoảng này để hình gọn và dễ đọc hơn. "
    "Timeline vẫn vẽ toàn audio và tô nền vùng được chọn."
))
display(Audio(interval_audio, rate=sr))

,start,end,duration,score,salience_mean,velocity_peak_norm,ats_amplitude,salience_amplitude,tension_mean,instability_mean,reason
0,1.25,8.25,7.0,0.787155,0.699249,1.0,0.770982,0.719422,0.581024,0.735362,Đoạn được chọn vì có tổng hợp tốt giữa dynamic...


### Khoảng minh họa đề xuất: **1.25s – 8.25s**
Đoạn được chọn vì có tổng hợp tốt giữa dynamic salience, chuyển trạng thái nhanh và biên độ biến thiên A/T/S. Đây là vùng phù hợp để minh họa bằng timeline và 3D trajectory.

Các hình 3D phía dưới sẽ dùng khoảng này để hình gọn và dễ đọc hơn. Timeline vẫn vẽ toàn audio và tô nền vùng được chọn.

In [ ]:
# ============================================================
# CELL 12 — Hình 1/3: 3D context A/T/(1-S), tô nổi bật khoảng minh họa
# ============================================================
def _largest_transition_idx(df: pd.DataFrame) -> int:
    if len(df) == 0 or 'state_velocity' not in df.columns:
        return 0
    return int(np.nanargmax(df['state_velocity'].to_numpy(dtype=float)))


def _plotly_html_path(out_dir: Path, prefix: str, suffix: str) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f'{prefix}_{suffix}.html'


def _make_hover_text(df: pd.DataFrame, use_instability: bool = True) -> List[str]:
    texts = []
    for r in df.itertuples():
        base = (
            f"t={r.center_sec:.2f}s"
            f"<br>A={r.activation_score_smooth:.3f}"
            f"<br>T={r.tension_score_smooth:.3f}"
            f"<br>S={r.stability_score_smooth:.3f}"
            f"<br>Salience={r.dynamic_salience:.3f}"
            f"<br>Velocity={r.state_velocity:.3f}"
        )
        if use_instability:
            base += f"<br>Instability={r.instability_score_smooth:.3f}"
        texts.append(base)
    return texts


def _add_key_markers(fig: go.Figure, df: pd.DataFrame, x_col: str, y_col: str, z_col: str, idx: Optional[int] = None):
    if len(df) == 0:
        return
    if idx is None:
        idx = _largest_transition_idx(df)
    idx = int(np.clip(idx, 0, len(df)-1))
    markers = [
        ('Start', 0, 'circle', '#2B8A3E', 9),
        ('End', len(df)-1, 'x', '#C92A2A', 10),
        ('Largest transition', idx, 'cross', '#7B2CBF', 9),
    ]
    for name, i, symbol, color, size in markers:
        r = df.iloc[i]
        fig.add_trace(go.Scatter3d(
            x=[r[x_col]], y=[r[y_col]], z=[r[z_col]],
            mode='markers+text',
            text=[name],
            textposition='top center',
            marker=dict(size=size, color=color, symbol=symbol, line=dict(width=2, color='white')),
            name=name,
            hovertext=[f"{name}<br>t={r['center_sec']:.2f}s<br>A={r['activation_score_smooth']:.3f}<br>T={r['tension_score_smooth']:.3f}<br>S={r['stability_score_smooth']:.3f}<br>Salience={r['dynamic_salience']:.3f}"],
            hoverinfo='text',
        ))


def _common_scene_layout(x_title: str, y_title: str, z_title: str, camera_eye=None):
    if camera_eye is None:
        camera_eye = dict(x=1.65, y=1.55, z=1.18)
    return dict(
        xaxis=dict(title=x_title, range=[0, 1], backgroundcolor='rgb(248,249,250)', gridcolor='rgb(215,220,225)'),
        yaxis=dict(title=y_title, range=[0, 1], backgroundcolor='rgb(248,249,250)', gridcolor='rgb(215,220,225)'),
        zaxis=dict(title=z_title, range=[0, 1], backgroundcolor='rgb(248,249,250)', gridcolor='rgb(215,220,225)'),
        aspectmode='cube',
        camera=dict(eye=camera_eye),
    )


def plot_ati_context_interactive(full_df: pd.DataFrame, selected_df: pd.DataFrame, out_dir: Path, prefix: str, interval: Tuple[float, float]):
    """Hình 1: toàn bộ audio là nền xám, khoảng được chọn là đường màu nổi bật trong A/T/(1-S)."""
    fig = go.Figure()

    # Full-audio context as faint gray markers.
    fig.add_trace(go.Scatter3d(
        x=full_df['activation_score_smooth'],
        y=full_df['tension_score_smooth'],
        z=full_df['instability_score_smooth'],
        mode='markers',
        marker=dict(size=3.2, color='rgba(120,120,120,0.22)'),
        name='All audio windows',
        hovertext=_make_hover_text(full_df),
        hoverinfo='text',
        showlegend=True,
    ))

    # Selected interval: colored line + markers.
    fig.add_trace(go.Scatter3d(
        x=selected_df['activation_score_smooth'],
        y=selected_df['tension_score_smooth'],
        z=selected_df['instability_score_smooth'],
        mode='lines+markers',
        line=dict(width=7, color='rgba(80,80,80,0.62)'),
        marker=dict(
            size=6 + 12 * selected_df['dynamic_salience'],
            color=selected_df['center_sec'], colorscale='Viridis',
            colorbar=dict(title='Time (s)'),
            opacity=0.96,
            line=dict(width=0.8, color='white'),
        ),
        name=f'Selected interval {interval[0]:.2f}s–{interval[1]:.2f}s',
        hovertext=_make_hover_text(selected_df),
        hoverinfo='text',
    ))

    _add_key_markers(fig, selected_df, 'activation_score_smooth', 'tension_score_smooth', 'instability_score_smooth')

    # High-salience reference corner.
    fig.add_trace(go.Scatter3d(
        x=[1.0], y=[1.0], z=[1.0],
        mode='markers+text',
        text=['High salience corner'],
        textposition='bottom center',
        marker=dict(size=4, color='rgba(220, 53, 69, 0.30)', symbol='diamond'),
        name='High salience corner',
        hoverinfo='skip',
    ))

    fig.update_layout(
        title=f'3D Context in A/T/(1-S) Space — selected window {interval[0]:.2f}s–{interval[1]:.2f}s',
        scene=_common_scene_layout('Activation', 'Tension', 'Instability = 1 - Stability'),
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.72)'),
        margin=dict(l=0, r=0, b=0, t=50),
        height=720,
    )

    html_path = _plotly_html_path(out_dir, prefix, '01_interactive_3d_context_ATI')
    fig.write_html(str(html_path), include_plotlyjs='cdn')
    print('Saved interactive HTML:', html_path)
    fig.show()
    return html_path

p_context_html = plot_ati_context_interactive(ats_df, selected_df, out_dir, prefix, (best_start, best_end))


Saved interactive HTML: /content/drive/MyDrive/motionemo_v10_ats_3d_upload_demo_optimized/calm_but_abit_hesitation_ats3d_optimized/calm_but_abit_hesitation_01_interactive_3d_context_ATI.html


In [ ]:
# ============================================================
# CELL 13 — Hình 2/3: 3D Activation–Tension–Stability State Trajectory
# ============================================================
def plot_ats_state_interactive(df: pd.DataFrame, out_dir: Path, prefix: str, interval: Tuple[float, float]):
    """Hình 2: quỹ đạo trực tiếp trong không gian A/T/S có thể diễn giải."""
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=df['activation_score_smooth'],
        y=df['tension_score_smooth'],
        z=df['stability_score_smooth'],
        mode='lines+markers',
        line=dict(width=7, color='rgba(95,95,95,0.60)'),
        marker=dict(
            size=6 + 12 * df['dynamic_salience'],
            color=df['center_sec'], colorscale='Viridis',
            colorbar=dict(title='Time (s)'),
            opacity=0.96,
            line=dict(width=0.8, color='white'),
        ),
        text=_make_hover_text(df, use_instability=False),
        hoverinfo='text',
        name='A/T/S trajectory',
    ))

    _add_key_markers(fig, df, 'activation_score_smooth', 'tension_score_smooth', 'stability_score_smooth')

    fig.update_layout(
        title=f'3D A/T/S State Trajectory — selected window {interval[0]:.2f}s–{interval[1]:.2f}s',
        scene=_common_scene_layout('Activation', 'Tension', 'Stability', camera_eye=dict(x=1.70, y=1.45, z=1.15)),
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.72)'),
        margin=dict(l=0, r=0, b=0, t=50),
        height=720,
    )

    html_path = _plotly_html_path(out_dir, prefix, '02_interactive_3d_ATS_state_trajectory')
    fig.write_html(str(html_path), include_plotlyjs='cdn')
    print('Saved interactive HTML:', html_path)
    fig.show()
    return html_path

p_ats_html = plot_ats_state_interactive(selected_df, out_dir, prefix, (best_start, best_end))


Saved interactive HTML: /content/drive/MyDrive/motionemo_v10_ats_3d_upload_demo_optimized/calm_but_abit_hesitation_ats3d_optimized/calm_but_abit_hesitation_02_interactive_3d_ATS_state_trajectory.html


In [ ]:
# ============================================================
# CELL 14 — Hình 3/3: 3D A/T/(1-S) Salience Cube
# ============================================================
def plot_ati_salience_interactive(df: pd.DataFrame, out_dir: Path, prefix: str, interval: Tuple[float, float]):
    """Hình 3: A/T/(1-S) để cả ba trục cùng tăng theo mức nổi bật động học."""
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=df['activation_score_smooth'],
        y=df['tension_score_smooth'],
        z=df['instability_score_smooth'],
        mode='lines+markers',
        line=dict(width=7, color='rgba(95,95,95,0.62)'),
        marker=dict(
            size=7 + 14 * df['dynamic_salience'],
            color=df['dynamic_salience'], colorscale='Viridis',
            colorbar=dict(title='Dynamic<br>salience'),
            opacity=0.96,
            line=dict(width=0.8, color='white'),
        ),
        text=_make_hover_text(df),
        hoverinfo='text',
        name='A/T/(1-S) trajectory',
    ))

    _add_key_markers(fig, df, 'activation_score_smooth', 'tension_score_smooth', 'instability_score_smooth')

    # Diagonal reference: low salience to high salience corner.
    fig.add_trace(go.Scatter3d(
        x=[0, 1], y=[0, 1], z=[0, 1],
        mode='lines',
        line=dict(width=3, color='rgba(220,53,69,0.35)', dash='dash'),
        name='Low → high salience diagonal',
        hoverinfo='skip',
    ))
    fig.add_trace(go.Scatter3d(
        x=[1.0], y=[1.0], z=[1.0],
        mode='markers+text',
        text=['High salience corner'],
        textposition='bottom center',
        marker=dict(size=5, color='rgba(220, 53, 69, 0.35)', symbol='diamond'),
        name='High salience corner',
        hoverinfo='skip',
    ))

    fig.update_layout(
        title=f'3D A/T/(1-S) Salience Cube — selected window {interval[0]:.2f}s–{interval[1]:.2f}s',
        scene=_common_scene_layout('Activation', 'Tension', 'Instability = 1 - Stability', camera_eye=dict(x=1.55, y=1.60, z=1.20)),
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.72)'),
        margin=dict(l=0, r=0, b=0, t=50),
        height=720,
        annotations=[dict(
            text='Instability is used so all three axes increase toward stronger dynamic salience.',
            x=0.5, y=0.02, xref='paper', yref='paper', showarrow=False,
            font=dict(size=12, color='#495057')
        )]
    )

    html_path = _plotly_html_path(out_dir, prefix, '03_interactive_3d_ATI_salience_cube')
    fig.write_html(str(html_path), include_plotlyjs='cdn')
    print('Saved interactive HTML:', html_path)
    fig.show()
    return html_path

p_ati_html = plot_ati_salience_interactive(selected_df, out_dir, prefix, (best_start, best_end))


Saved interactive HTML: /content/drive/MyDrive/motionemo_v10_ats_3d_upload_demo_optimized/calm_but_abit_hesitation_ats3d_optimized/calm_but_abit_hesitation_03_interactive_3d_ATI_salience_cube.html
